<a href="https://colab.research.google.com/github/majozuu/Basic-Flutter-App/blob/main/notebooks/differentiable-parameterizations/appendix/infinite_patterns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
# Install BeautifulSoup for HTML parsing if it's not already installed
!pip install beautifulsoup4

In [42]:
import requests
from bs4 import BeautifulSoup

def get_grid_data_from_google_doc(doc_url):
    data_points = []
    try:
        response = requests.get(doc_url)
        response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
        content = response.text

        # Check if the content is HTML
        if "<html" in content.lower() and "<body" in content.lower():
            soup = BeautifulSoup(content, 'html.parser')

            # Attempt to find the main content div
            doc_content_div = soup.find('div', id='contents')
            if not doc_content_div:
                doc_content_div = soup.find('div', class_='doc-content') # Another common class
            if not doc_content_div:
                # Fallback to general body text if specific divs not found
                doc_content_div = soup.body if soup.body else soup

            # Extract text from the identified block
            extracted_text = doc_content_div.get_text(separator='\n', strip=True)
            raw_lines = extracted_text.splitlines()
        else:
            # Assume it's plain text already if not HTML
            raw_lines = content.splitlines()

        # Filter out empty lines and specific header/introductory lines
        lines_to_process = []
        header_identifiers = {"x-coordinate", "character", "y-coordinate"} # Case-insensitive
        # More generic intro text check
        intro_text_keywords = {"this is an example document", "the table below contains"}

        for line in raw_lines:
            stripped_line = line.strip().lower()
            if not stripped_line:
                continue
            if stripped_line in header_identifiers:
                continue
            if any(keyword in stripped_line for keyword in intro_text_keywords):
                continue

            lines_to_process.append(line.strip()) # Keep original casing for char, remove leading/trailing spaces


        # Process lines_to_process in triplets (x_coord, char, y_coord)
        for i in range(0, len(lines_to_process) - 2, 3):
            x_str = lines_to_process[i]
            char_val = lines_to_process[i+1]
            y_str = lines_to_process[i+2]

            try:
                # Validate that x_str and y_str are digits and char_val is not
                if x_str.isdigit() and y_str.isdigit() and not char_val.isdigit():
                    x = int(x_str)
                    y = int(y_str)
                    data_points.append((char_val, x, y))
                else:
                    # Print warnings for triplets that don't fit the expected pattern
                    print(f"Warning: Skipping malformed triplet: ({x_str}, {char_val}, {y_str})")
            except ValueError as e:
                print(f"Warning: Error parsing triplet ({x_str}, {char_val}, {y_str}) - {e}")

    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from URL {doc_url}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    return data_points

def print_grid_from_data(data):
    if not data:
        print("No grid data to display.")
        return

    max_x = 0
    max_y = 0
    char_map = {}

    for char, x, y in data:
        max_x = max(max_x, x)
        max_y = max(max_y, y)
        char_map[(x, y)] = char

    # Initialize grid with spaces, dimensions based on max_x and max_y
    grid_width = max_x + 1
    grid_height = max_y + 1
    grid = [[' ' for _ in range(grid_width)] for _ in range(grid_height)]

    # Populate grid
    for (x, y), char in char_map.items():
        # Ensure coordinates are within grid bounds before placing character
        if 0 <= y < grid_height and 0 <= x < grid_width:
            grid[y][x] = char

def display_unicode_grid(doc_url):
    print(f"Attempting to fetch and process data from: {doc_url}")
    grid_data = get_grid_data_from_google_doc(doc_url)

    if grid_data:
        print(f"Successfully parsed {len(grid_data)} data points.")
        print_grid_from_data(grid_data)
    else:
        print("No valid data points were parsed. Please ensure the Google Doc content is structured as 'X_COORD_VALUE', 'CHARACTER_VALUE', 'Y_COORD_VALUE' on consecutive lines after headers.")

In [45]:
# Define the Google Doc URL
doc_url = 'https://docs.google.com/document/d/e/2PACX-1vSvM5gDlNvt7npYHhp_XfsJvuntUhq184By5xO_pA4b_gCWeXb6dM6ZxwN8rE6S4ghUsCj2VKR21oEP/pub'

# Call the main function
display_unicode_grid(doc_url)


Attempting to fetch and process data from: https://docs.google.com/document/d/e/2PACX-1vSvM5gDlNvt7npYHhp_XfsJvuntUhq184By5xO_pA4b_gCWeXb6dM6ZxwN8rE6S4ghUsCj2VKR21oEP/pub
Successfully parsed 340 data points.
